# 3.3 · Non-Functional Requirements for AI Agents

### A Practical Guide to Cost, Compute Footprint, and Environmental Impact

## Overview

When building software, requirements come in two flavors:

- **Functional requirements** - *what* the system does ("summarize a document", "answer a question").
- **Non-functional requirements (NFRs)** - *how well* it does it: speed, cost, reliability, and resource usage.

For traditional software, NFRs are well understood: measure CPU, memory, and response time; profile hot paths; right-size infrastructure. For AI agents, the picture is more complex, and the stakes are higher.

### Why agents amplify NFR concerns

An AI agent is not a single LLM call. It is an **autonomous loop** that plans, calls tools, observes results, and iterates until a task is complete. Each loop iteration is a new LLM call. Each tool result is appended to the context window and re-sent on the next call. A task that looks like "one request" from the outside can silently generate 5–20 LLM calls under the hood.

That loop changes the NFR calculus entirely. Costs that were predictable per-call become compounding across turns. Latency that was one round-trip becomes N sequential round-trips. Memory that was stateless now carries a growing conversation history. These are not edge cases; they are the default behavior of any non-trivial agent.

### The NFR landscape for AI agents

Below is an orientation to the key dimensions. This notebook covers the first three in depth with working code; the remaining sections introduce each topic and point you to where you can go deeper.

---

**Cost**

LLM APIs charge per token, separately for input and output. In an agent, input token costs compound: every turn re-sends the full conversation history, so a 5-step agent can cost 3–5× more than a naive per-call estimate would suggest. Cost is usually the first NFR to cause a production incident, because it is invisible until it isn't.

Key metrics: USD per task, cost per thousand requests, input-to-output token ratio.

---

**Latency & Throughput**

Latency for a single LLM call has two components: time-to-first-token (dominated by prompt length) and generation time (proportional to output length). In an agent, these stack: a 4-step task with 1.5 s LLM calls and 0.5 s tool calls takes ≥ 8 seconds end-to-end regardless of how fast any individual step is. Throughput, tokens per second, determines how many concurrent requests the system can sustain.

Key metrics: end-to-end task latency (p50/p95), time-to-first-token, tokens/second.

---

**Environmental Footprint**

A single LLM query uses roughly 0.001–0.003 kWh of energy. An agent completing a 5-step task uses 5–15× that. At scale, this becomes a meaningful operational and reputational consideration. The carbon intensity depends heavily on where the inference runs: renewable-powered data centres in Iceland or the US West Coast carry a fraction of the footprint of coal-heavy grids.

Key metrics: kWh per task, estimated CO₂e per thousand requests.

---

**Agent Efficiency**

Beyond raw cost and latency, it is worth asking whether the agent is taking the *right* number of steps. An agent that calls the same tool twice, backtracks unnecessarily, or generates redundant reasoning is wasting both compute and money. Tracking LLM invocation count per task, tool call count, and re-call rate per tool gives you a signal on reasoning quality that token counts alone do not.

Key metrics: LLM invocations per task, tool calls per task, redundant tool call rate, task completion rate.

---

**Scalability & Multi-tenancy**

A single-user agent is very different from one serving thousands of concurrent users. At scale, you need to think about: how many parallel agent sessions can your inference endpoint sustain? How do you isolate one tenant's conversation history from another's? Does your context management strategy hold when 500 agents are all accumulating history simultaneously? These questions shape whether you deploy agents as stateless workers behind a queue, or as long-lived processes with session affinity.

Key metrics: requests per second at target latency, context isolation guarantees, memory footprint under concurrent load.

---

**Concurrency & Shared Memory**

Related to scalability is the question of *within-agent* concurrency. ReAct-style agents are sequential by default, but many tasks have independent sub-steps that could run in parallel. Beyond single-agent parallelism, multi-agent systems introduce shared memory, the idea that agents can read from and write to a common context or knowledge store. Providers including Anthropic and OpenAI have recently introduced memory APIs and cross-session persistence primitives that make this increasingly practical. How you design for shared state under concurrent access has direct implications for consistency, cost, and security.

Key metrics: parallel tool call utilization, shared memory read/write latency, cross-agent context overhead.

---

**Security**

Agents that call tools, read files, browse the web, or execute code introduce a new class of attack surface. Prompt injection, where malicious content in a tool result hijacks the agent's reasoning, is the most common. Agents with persistent memory or access to external systems raise additional questions around data leakage, authorization scope, and audit trails. Security is an NFR that is easy to defer and expensive to retrofit.

For a practical guide to agent security patterns, see the companion recipe: [AI Agent Security](../AI_Agent_Security/AI_Agent_Security_Cookbook.ipynb).

---

**Maintainability & Observability**

An agent that works in a notebook is not the same as one that works reliably in production. Observability, the ability to see what an agent did, why it did it, and where it spent time and money, is what separates a prototype from a system you can operate and improve. Structured tracing captures every LLM call, tool invocation, and latency breakdown in a queryable log. Without it, debugging a misbehaving agent is guesswork.

For end-to-end tracing with Langfuse, see the companion recipe: [Observability & Tracing](../Tracing/Tracing_Agent.ipynb).

---

### What this notebook covers

This notebook is a practical guide for engineers building or operating AI agents in production. The goal is not to make NFRs feel like a compliance exercise; it is to give you the intuition and the tooling to ask the right questions early, before a surprise bill or a latency regression surfaces in production.

We focus on the first three dimensions above, where hands-on measurement makes the most immediate difference:

1. **Cost**: token economics, how to measure real spend on a running agent, and strategies to reduce it without compromising quality.
2. **Compute footprint**: latency and throughput, what drives them, and how to reason about them when choosing models and infrastructure.
3. **Environmental footprint**: an informed, grounded perspective on energy and carbon impact.

Each section pairs explanation with working code you can run against your own watsonx.ai environment. By the end you will have a **reusable `NFRProfiler` class** that instruments cost, latency, and memory in a single dashboard, ready to drop into any LangChain project.

Code examples use IBM Granite models via watsonx.ai. **Prerequisites:** Python 3.11+, basic familiarity with LangChain/LangGraph.

## Table of Contents

- [Overview: The NFR Landscape for AI Agents](#overview)
  - Cost · Latency & Throughput · Environmental Footprint · Agent Efficiency · Scalability · Concurrency · Security · Observability

1. [Setup & Environment](#section-1)
2. [Cost: Understanding Token Economics](#section-2)
3. [Cost: Measuring Agent Spend in Practice](#section-3)
4. [Cost: Optimization Strategies](#section-4)
5. [Compute Footprint: Latency & Throughput](#section-5)
6. [Compute Footprint: Memory & Context Management](#section-6)
7. [Environmental Footprint](#section-7)
8. [Summary & Reusable NFR Profiler](#section-8)

<a id='section-1'></a>
## 1. Setup & Environment

You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

To avoid Python package dependency conflicts, we recommend setting up a [virtual environment](https://docs.python.org/3/library/venv.html).

This notebook is compatible with Python 3.12 and Python 3.11.

### 1.1 Install dependencies

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install \
    "git+https://github.com/ibm-granite-community/utils.git" \
    langgraph \
    langchain \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    pandas \
    matplotlib \
    seaborn \
    rich \
    psutil \
    requests \
    tiktoken
! echo "::endgroup::"

### 1.2 Import libraries

In [ ]:
import os
import time
import json
import warnings
import psutil
from dataclasses import dataclass, field
from typing import Any, Optional

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from rich import print as rprint

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.callbacks import UsageMetadataCallbackHandler

warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid", palette="muted")

def get_memory_mb() -> float:
    """Return current process RSS memory usage in megabytes."""
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

print("Imports successful")

### 1.3 Initialize the Granite model

This notebook uses a **Granite 4.2 model** served by the [Replicate](https://replicate.com) cloud service. See [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb) for information on setting up Replicate access and obtaining an [API token](https://replicate.com/account/api-tokens). You will need the `REPLICATE_API_TOKEN` environment variable set.

The `get_llm()` helper below is the same one used in the other workshop notebooks. We call it with `backend="replicate"` so every measurement in this notebook comes from the same model and provider.

In [ ]:
import os

import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

In [ ]:
llm = get_llm(backend="replicate", repetition_penalty=1.05)

# Cap the output length: keeps runs fast and cheap, and makes token counts comparable across runs.
llm.model_kwargs["max_completion_tokens"] = 300

rprint(f"[green]Connected via {type(llm).__name__}, model: {llm.model}[/green]")

<a id='section-2'></a>
## 2. Cost: Understanding Token Economics

### What is a token?

LLMs do not process text character-by-character or word-by-word, they process **tokens**, which are sub-word units produced by a tokenizer. As a rule of thumb:

> **1 token ≈ 0.75 English words** (roughly 4 characters)

So "The quick brown fox" is approximately 5 tokens, and a 500-word essay is roughly 670 tokens. The exact mapping depends on the model's vocabulary; you can browse [Granite's tokenizer vocabulary on HuggingFace](https://huggingface.co/ibm-granite/granite-3.3-8b-instruct/blob/main/tokenizer.json) to get a concrete feel for how text is split.

### How pricing works

LLM API providers charge separately for **input tokens** (what you send) and **output tokens** (what the model generates). Output tokens are typically more expensive because generating each token requires a full forward pass through the model.

Pricing varies significantly across providers and model tiers, and changes frequently; always check your provider's current pricing page before budgeting. As a general orientation, costs typically range from a fraction of a cent to several dollars per million tokens, with smaller/distilled models at the low end and large frontier models at the high end. Pricing is almost always quoted **per million tokens** for both input and output separately.

The table below uses illustrative figures to demonstrate the *shape* of the difference across tiers. Treat the numbers as approximations, not commitments.

| Model tier | Input ($/1M tokens) | Output ($/1M tokens) |
|---|---|---|
| Small / efficient | ~$0.25 | ~$1.25 |
| Medium | ~$3 | ~$15 |
| Large / flagship | ~$15 | ~$75 |

### The agent cost multiplier

In a multi-step agent, the **entire conversation history** is re-sent to the model on every turn. If your agent takes 5 steps and each step accumulates 200 tokens of context:

- Turn 1: 200 input tokens  
- Turn 2: 400 input tokens (history doubles)  
- Turn 3: 600 input tokens  
- Turn 4: 800 input tokens  
- Turn 5: 1000 input tokens  
- **Total input: 3000 tokens** (not 1000)

This **quadratic growth** is why a 5-step agent can cost 3–5× more than a single call with the same total content.

In [ ]:
# TokenCostEstimator: reusable utility for cost calculations

@dataclass
class TokenCostEstimator:
    """Estimates LLM API cost from token counts."""
    input_price_per_m: float    # USD per 1M input tokens
    output_price_per_m: float   # USD per 1M output tokens
    model_label: str = "unknown"

    def estimate(self, input_tokens: int, output_tokens: int) -> dict:
        cost = (
            (input_tokens / 1_000_000) * self.input_price_per_m
            + (output_tokens / 1_000_000) * self.output_price_per_m
        )
        return {
            "model": self.model_label,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
            "estimated_cost_usd": round(cost, 6),
        }

# Illustrative pricing for three tiers
SMALL_MODEL  = TokenCostEstimator(0.25,  1.25,  "granite-small")
MEDIUM_MODEL = TokenCostEstimator(3.0,   15.0,  "granite-medium")
LARGE_MODEL  = TokenCostEstimator(15.0,  75.0,  "granite-large")

# Demonstrate cost across a typical range of token counts
# Each tuple is: (scenario label, input_tokens, output_tokens)
scenarios = [
    ("Single chat turn",       150,   80),
    ("5-step agent task",     1200,  400),
    ("Complex research task", 5000,  800),
]

rows = []
for label, inp, out in scenarios:
    for estimator in [SMALL_MODEL, MEDIUM_MODEL, LARGE_MODEL]:
        result = estimator.estimate(inp, out)
        rows.append({"scenario": label, **result})

df_cost = pd.DataFrame(rows)
pivot = df_cost.pivot(index="scenario", columns="model", values="estimated_cost_usd")

# Reorder rows from simple to complex
scenario_order = ["Single chat turn", "5-step agent task", "Complex research task"]
pivot = pivot.reindex(scenario_order)

ax = pivot.plot(kind="bar", figsize=(10, 5), rot=0, colormap="viridis")
ax.set_title("Estimated cost by scenario and model tier", fontsize=14, pad=12)
ax.set_xlabel("Scenario")
ax.set_ylabel("Estimated cost (USD)")
ax.legend(title="Model tier")
for container in ax.containers:
    ax.bar_label(container, fmt="$%.5f", fontsize=7, padding=3)
plt.tight_layout()
plt.show()

rprint(pivot.to_string())

### Context growth: why agent costs compound

As the agent iterates, the **entire conversation history is re-sent on every turn**. The chart below illustrates this: if each turn adds 150 new tokens and there is a 200-token system prompt, the input token count grows linearly across turns, but the *cumulative* total grows quadratically, because earlier turns are re-paid on every subsequent call.

```
Turn 1: 350 tokens sent   (200 system + 150 new)
Turn 2: 500 tokens sent   (200 system + 300 accumulated)
Turn 3: 650 tokens sent   (200 system + 450 accumulated)
...
Turn 6: 1100 tokens sent  (200 system + 900 accumulated)
──────────────────────────────────────────
Total input: 3,750 tokens  ≠  6 × 150 = 900 (naive estimate)
Cost multiplier: ~4.2×
```

This compounding effect is why cost estimations based on a single-call token count routinely underestimate production spend by 3–5×. The mitigation strategies in Section 4 directly target this curve.

![Context growth chart: input tokens per turn grow linearly; cumulative cost diverges from the naive flat estimate by turn 3 onward](context_growth_illustration.png)

<a id='section-3'></a>
## 3. Cost: Measuring Agent Spend in Practice

Understanding the theory is useful, but what you really need in production is **instrumentation** - the ability to observe actual token counts and costs on every agent run.

LangChain provides `UsageMetadataCallbackHandler` (from `langchain_core`) which intercepts LLM calls and accumulates token usage. It works with any provider that returns usage metadata, including IBM watsonx.

We will build a minimal agent with one tool and measure its token footprint across queries of varying complexity.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# A simple calculator tool (no external API needed)
@tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression and returns the result as a string.
    Only supports basic arithmetic: +, -, *, /, ** and parentheses.

    Args:
        expression: A mathematical expression, e.g. '(3 + 4) * 2'

    Returns:
        The numeric result as a string.
    """
    import ast
    try:
        tree = ast.parse(expression, mode="eval")
        allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant,
                   ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Pow,
                   ast.USub, ast.UAdd)
        for node in ast.walk(tree):
            if not isinstance(node, allowed):
                return "Error: unsupported operation"
        return str(eval(compile(tree, "<string>", "eval")))
    except Exception as e:
        return f"Error: {e}"

agent = create_react_agent(llm, tools=[calculator])
rprint("[green]ReAct agent created with calculator tool[/green]")

In [ ]:
@dataclass
class RunMetrics:
    prompt: str
    input_tokens: int
    output_tokens: int
    total_tokens: int
    estimated_cost_usd: float
    latency_ms: float
    llm_invocations: int

def run_agent_measured(prompt: str, estimator: TokenCostEstimator) -> RunMetrics:
    """Run the agent and collect token usage, latency, and invocation count."""
    cb = UsageMetadataCallbackHandler()
    start = time.perf_counter()
    agent.invoke({"messages": [HumanMessage(content=prompt)]}, config={"callbacks": [cb]})
    elapsed_ms = (time.perf_counter() - start) * 1000

    input_tokens  = sum(v.get("input_tokens", 0)  for v in cb.usage_metadata.values())
    output_tokens = sum(v.get("output_tokens", 0) for v in cb.usage_metadata.values())
    total_tokens  = input_tokens + output_tokens
    # Each key in usage_metadata corresponds to one LLM call
    llm_invocations = len(cb.usage_metadata)

    cost_info = estimator.estimate(input_tokens, output_tokens)
    return RunMetrics(
        prompt=prompt,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        estimated_cost_usd=cost_info["estimated_cost_usd"],
        latency_ms=elapsed_ms,
        llm_invocations=llm_invocations,
    )

# Single run example
test_prompt = "What is (17 * 23) + (144 / 12)?"
metrics = run_agent_measured(test_prompt, SMALL_MODEL)

rprint(f"\nPrompt: [italic]{test_prompt}[/italic]")
rprint(f"  Input tokens:     [bold]{metrics.input_tokens}[/bold]")
rprint(f"  Output tokens:    [bold]{metrics.output_tokens}[/bold]")
rprint(f"  Total tokens:     [bold]{metrics.total_tokens}[/bold]")
rprint(f"  Estimated cost:   [bold]${metrics.estimated_cost_usd:.6f}[/bold]")
rprint(f"  Latency:          [bold]{metrics.latency_ms:.0f} ms[/bold]")
rprint(f"  LLM invocations:  [bold]{metrics.llm_invocations}[/bold]")

In [ ]:
benchmark_prompts = [
    ("Simple",   "What is 5 + 3?"),
    ("Medium",   "Calculate the compound interest on $10,000 at 5% annual rate over 3 years. Use (1 + 0.05) ** 3 * 10000 - 10000."),
    ("Complex",  "I need to plan a budget. My monthly income is $5,200. Rent is $1,500, utilities are $200, groceries are $350, transport is $180, and savings should be 20% of net (income minus fixed costs). Calculate: first find fixed costs (1500+200+350+180), then net = 5200 - fixed, then savings = net * 0.20, then discretionary = net - savings."),
]

results = []
for label, prompt in benchmark_prompts:
    m = run_agent_measured(prompt, SMALL_MODEL)
    results.append({
        "complexity": label,
        "input_tokens": m.input_tokens,
        "output_tokens": m.output_tokens,
        "cost_usd": m.estimated_cost_usd,
        "latency_ms": m.latency_ms,
        "llm_invocations": m.llm_invocations,
    })

df_bench = pd.DataFrame(results)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

colors = sns.color_palette("muted", 4)

# Panel 1: token breakdown (input vs output stacked)
axes[0].bar(df_bench["complexity"], df_bench["input_tokens"],  color=colors[0],
            label="Input tokens (history re-sent each turn)")
axes[0].bar(df_bench["complexity"], df_bench["output_tokens"], bottom=df_bench["input_tokens"],
            color=colors[1], label="Output tokens (new generation)")
axes[0].set_title("Token usage per run")
axes[0].set_ylabel("Tokens")
axes[0].legend(loc="upper left", fontsize=7)

# Panel 2: LLM invocation count (shows WHY input tokens grow
axes[1].bar(df_bench["complexity"], df_bench["llm_invocations"], color=colors[2])
axes[1].set_title("LLM calls per run\n(each call re-sends full history)")
axes[1].set_ylabel("# of LLM invocations")
for i, v in enumerate(df_bench["llm_invocations"]):
    axes[1].text(i, v + 0.05, str(int(v)), ha="center", fontsize=10, fontweight="bold")

# Panel 3: estimated cost
axes[2].bar(df_bench["complexity"], df_bench["cost_usd"], color=colors[3])
axes[2].set_title("Estimated cost (USD)")
axes[2].set_ylabel("USD")
for i, v in enumerate(df_bench["cost_usd"]):
    axes[2].text(i, v + v * 0.02, f"${v:.6f}", ha="center", fontsize=8)

# Panel 4: latency
axes[3].bar(df_bench["complexity"], df_bench["latency_ms"], color=colors[0])
axes[3].set_title("Latency (ms)")
axes[3].set_ylabel("Milliseconds")
for i, v in enumerate(df_bench["latency_ms"]):
    axes[3].text(i, v + 10, f"{v:.0f} ms", ha="center", fontsize=8)

plt.suptitle("Agent run metrics: simple vs medium vs complex", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

rprint(df_bench.to_string(index=False))

<a id='section-4'></a>
## 4. Cost: Optimisation Strategies

Once you can measure cost, you can reduce it. The five main levers are:

| Strategy | Mechanism | Typical savings |
|---|---|---|
| **Prompt compression** | Trim system prompt, summarise history | 20–50% of input tokens |
| **Model routing** | Use a small model for simple queries | 5–20× cost reduction |
| **Prompt caching** | Reuse identical prefix tokens across calls | 50–90% of repeated-context cost |
| **Output length control** | Set `max_completion_tokens`; use structured output | 30–60% output tokens |
| **Tool call batching** | Ask the model to call multiple tools in one turn | Reduces total turns |

We will implement and demonstrate the first two - the highest-impact strategies with no external dependencies. For a deeper treatment of context window management, including summarisation pipelines, retrieval-augmented truncation, and sliding window patterns. See the companion recipe: [Context Management](../ContextManagement/context_management.md).

### 4.1 Prompt compression

In a long conversation, the middle turns often contain resolved sub-tasks that no longer need to be verbatim in context. A **sliding window** keeps only the most recent N turns, replacing older content with a brief summary.

In [ ]:
def naive_token_count(text: str) -> int:
    """Rough token estimate: 1 token per ~4 characters.

    For exact counts use the provider's tokenize endpoint instead:
    - watsonx.ai: POST /ml/v1/text/tokenization  (returns token_count per input)
      https://cloud.ibm.com/apidocs/watsonx-ai#text-tokenization
    - OpenAI-compatible: tiktoken library  (cl100k_base for GPT-4 / Granite)
      https://github.com/openai/tiktoken
    """
    return max(1, len(text) // 4)

def count_messages_tokens(messages: list) -> int:
    return sum(naive_token_count(m.content) for m in messages)

def compress_history(
    messages: list,
    keep_last_n: int = 4,
    summary: str = "[Earlier conversation summarised: context established, goals clarified.]"
) -> list:
    """
    Retain the system message and the most recent `keep_last_n` messages.
    Older messages are replaced with a single summary placeholder.
    """
    system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
    non_system  = [m for m in messages if not isinstance(m, SystemMessage)]

    if len(non_system) <= keep_last_n:
        return messages

    summary_msg = AIMessage(content=summary)
    return system_msgs + [summary_msg] + non_system[-keep_last_n:]

# Build a synthetic 10-turn conversation history
synthetic_history = [
    SystemMessage(content="You are a helpful assistant specialized in financial analysis. Always show your calculations step-by-step."),
] + [
    msg for i in range(1, 6) for msg in [
        HumanMessage(content=f"Turn {i}: Calculate the ROI for a $10,000 investment with a ${i*200} annual return."),
        AIMessage(content=f"Turn {i} answer: ROI = ({i*200} / 10000) * 100 = {i*2}%. This represents a {'low' if i < 3 else 'moderate'} return."),
    ]
]

compressed = compress_history(synthetic_history, keep_last_n=4)

tokens_before = count_messages_tokens(synthetic_history)
tokens_after  = count_messages_tokens(compressed)
savings_pct   = (1 - tokens_after / tokens_before) * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(["Before compression", "After compression"],
              [tokens_before, tokens_after],
              color=sns.color_palette("muted", 2))
ax.bar_label(bars, labels=[f"{tokens_before} tokens", f"{tokens_after} tokens"], padding=4)
ax.set_title(f"Prompt compression: {savings_pct:.0f}% token reduction", fontsize=13)
ax.set_ylabel("Approximate token count")
plt.tight_layout()
plt.show()

rprint(f"Messages before: [bold]{len(synthetic_history)}[/bold] → after: [bold]{len(compressed)}[/bold]")
rprint(f"Token reduction: [bold]{tokens_before} → {tokens_after} ({savings_pct:.0f}% saved)[/bold]")

### 4.2 Model routing

Not every query requires a large, expensive model. A **router** classifies each incoming query by estimated complexity and dispatches it to the cheapest capable model. Simple arithmetic or fact-lookup queries go to the small model; open-ended reasoning or multi-step planning goes to the large one.

In [ ]:
def classify_complexity(query: str) -> str:
    """
    Heuristic complexity classifier.
    Returns 'simple' or 'complex'.

    In production, you would replace this with a lightweight classifier model
    or a small LLM call; the cost of that classification is far less than
    routing everything to the large model.
    """
    word_count = len(query.split())
    complex_keywords = {"compare", "analyse", "explain", "why", "how does",
                        "design", "strategy", "tradeoff", "plan", "evaluate"}
    has_complex_keyword = any(kw in query.lower() for kw in complex_keywords)
    has_multiple_steps  = query.count(".") >= 2 or query.count(";") >= 1

    if word_count > 35 or has_complex_keyword or has_multiple_steps:
        return "complex"
    return "simple"

def route_to_model(query: str) -> tuple[str, str]:
    """Returns (complexity_label, model_id)."""
    complexity = classify_complexity(query)
    if complexity == "simple":
        return "simple", "ibm/granite-4-h-small"
    return "complex", "ibm/granite-4-h-large"

# Simulate cost comparison: always-large vs routed
test_queries = [
    "What is 128 divided by 4?",
    "Convert 72 Fahrenheit to Celsius.",
    "What year was the Eiffel Tower built?",
    "Explain the tradeoffs between synchronous and asynchronous agent architectures in terms of latency, cost, and error handling.",
    "Design a cost optimisation strategy for an AI agent that processes 10,000 queries per day across three complexity tiers.",
]

rows = []
for q in test_queries:
    complexity, model_id = route_to_model(q)
    word_count = len(q.split())
    # Approximate token counts
    inp = naive_token_count(q) + 120   # query + system prompt overhead
    out = 60 + word_count // 2

    cost_routed = (SMALL_MODEL if complexity == "simple" else LARGE_MODEL).estimate(inp, out)["estimated_cost_usd"]
    cost_always_large = LARGE_MODEL.estimate(inp, out)["estimated_cost_usd"]

    rows.append({
        "query": q[:55] + ("..." if len(q) > 55 else ""),
        "routed_to": "small" if complexity == "simple" else "large",
        "cost_routed_usd": cost_routed,
        "cost_always_large_usd": cost_always_large,
        "savings_usd": cost_always_large - cost_routed,
    })

df_routing = pd.DataFrame(rows)

total_savings  = df_routing["savings_usd"].sum()
total_routed   = df_routing["cost_routed_usd"].sum()
total_unrouted = df_routing["cost_always_large_usd"].sum()
savings_pct    = (total_savings / total_unrouted) * 100

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(df_routing))
width = 0.35
ax.bar([i - width/2 for i in x], df_routing["cost_always_large_usd"], width,
       label="Baseline: always use large model (worst case)", color=sns.color_palette("muted")[3])
ax.bar([i + width/2 for i in x], df_routing["cost_routed_usd"], width,
       label="Routed: small model for simple queries, large for complex", color=sns.color_palette("muted")[0])
ax.set_xticks(list(x))
ax.set_xticklabels([r["query"][:30] + "..." for r in rows], rotation=15, ha="right", fontsize=8)
ax.set_ylabel("Estimated cost (USD)")
ax.set_title(f"Model routing: {savings_pct:.0f}% cost reduction on this query mix", fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

rprint(df_routing[["query", "routed_to", "cost_routed_usd", "cost_always_large_usd", "savings_usd"]].to_string(index=False))
rprint(f"\nTotal savings over {len(test_queries)} queries: [bold]${total_savings:.6f} ({savings_pct:.0f}%)[/bold]")

### 4.3 Output length control and prompt caching

**Output length control** is the simplest lever: set `max_completion_tokens` to prevent runaway generation. Pair it with structured output (JSON schemas, Pydantic models) so the model produces compact, machine-readable responses rather than lengthy prose.

**Prompt caching** is a provider-level feature where a long, stable prefix (system prompt, reference documents, tool definitions) is cached server-side. Subsequent calls that share that prefix pay only for the non-cached portion. IBM watsonx and other providers expose this via the API. The key design principle: **put the stable content first, variable content last**.

For an end-to-end observability view of cost, latency, and token usage in production, see the companion notebook: [Observability & Tracing with Langfuse](../Tracing/Tracing_Agent.ipynb).

<a id='section-5'></a>
## 5. Compute Footprint: Latency & Throughput

### Key concepts

- **Latency** is how long the user (or system) waits for a response. For LLMs it breaks into:
  - **Time-to-first-token (TTFT)**: the delay before any output begins. Dominated by prompt processing (pre-fill).
  - **Generation time**: the time to produce all output tokens. Scales linearly with output length (decode). The inverse, tokens generated per second, is the **inter-token latency**, which directly determines how fluid streamed responses feel to a user.
- **Throughput** is how fast the model generates output, measured in **tokens per second (tok/s)**.

### Why latency metrics inform production decisions

Understanding TTFT and inter-token latency is not just an academic exercise; it shapes every layer of a production agent deployment:

- **Model choice**: Smaller, distilled models have lower TTFT and higher tok/s but may require more turns to complete complex tasks, trading per-call speed for total task time. The right choice depends on your p95 latency SLO.
- **GPU selection**: TTFT is dominated by the pre-fill pass, which is memory-bandwidth-bound. Inter-token latency is compute-bound. A high-bandwidth-memory GPU (e.g., H100) reduces both, but the cost per query must be weighed against a managed API.
- **Replication and batching**: To sustain throughput under concurrent load you either increase replica count (horizontal scale) or increase batch size per GPU. Batching improves throughput but increases individual request latency, a tradeoff to model explicitly at your expected QPS.
- **User experience**: TTFT governs perceived responsiveness; inter-token latency governs perceived fluency. Streaming responses mitigate the impact of high generation time, but cannot compensate for a slow TTFT. A 3-second TTFT feels broken even if tokens then arrive at 80 tok/s.

### Why agents are latency-sensitive

Each agent turn is a sequential LLM call. Tool calls add their own latency (network round-trip, execution). A 4-step agent task with 1.5 s per LLM call and 0.5 s tool execution time takes **≥ 8 seconds** end-to-end, even if each individual step feels fast.

```
User ──► Turn 1 (1.5 s) ──► Tool A (0.5 s) ──► Turn 2 (1.5 s) ──► Tool B (0.5 s)
      ──► Turn 3 (1.5 s) ──► Tool C (0.5 s) ──► Turn 4 (1.5 s) ──► Answer
                                                              Total: ~8 s
```

Key levers to reduce latency:
1. Smaller or distilled models (lower TTFT, faster generation)
2. Streaming (show partial output while generation continues)
3. Parallel tool calls (call multiple tools simultaneously when dependencies allow)
4. Shorter prompts (less pre-fill work)

### 5.1 Measuring latency vs. prompt length

The first block below defines a `LatencyResult` dataclass and a `measure_latency` helper that times a raw LLM call and reads the token count from the response metadata. We then run three prompts: short, medium, and long to observe how pre-fill length drives wall-clock latency and throughput.

In [ ]:
@dataclass
class LatencyResult:
    prompt_label: str
    total_ms: float
    tokens_generated: int

    @property
    def tokens_per_second(self) -> float:
        if self.total_ms == 0:
            return 0.0
        return self.tokens_generated / (self.total_ms / 1000)

def measure_latency(prompt: str, label: str) -> LatencyResult:
    """Invoke the LLM and measure wall-clock latency."""
    start = time.perf_counter()
    response = llm.invoke(prompt)
    elapsed_ms = (time.perf_counter() - start) * 1000
    tokens_out = (
        response.usage_metadata.get("output_tokens", None)
        if hasattr(response, "usage_metadata") and response.usage_metadata
        else len(response.content.split())
    )
    return LatencyResult(label, round(elapsed_ms, 1), tokens_out or 30)


latency_prompts = [
    ("Short (10 words)",   "What is the capital of France?"),
    ("Medium (30 words)",  "Explain in two sentences why prompt length affects LLM latency. Consider both the pre-fill phase and the auto-regressive generation phase."),
    ("Long (70 words)",    "I am building an AI agent pipeline for customer support. My users ask questions about order status, returns, and product specifications. The agent needs to call three separate internal APIs, aggregate results, and produce a structured JSON response within 3 seconds. Given these constraints, what approach would you recommend for minimising latency while maintaining accuracy? Please consider model size, prompt design, and tool call parallelism."),
]

latency_results = [measure_latency(prompt, label) for label, prompt in latency_prompts]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels    = [r.prompt_label for r in latency_results]
latencies = [r.total_ms for r in latency_results]
tps_vals  = [r.tokens_per_second for r in latency_results]

colors = sns.color_palette("muted", 3)
axes[0].bar(labels, latencies, color=colors)
axes[0].set_title("Total latency by prompt length")
axes[0].set_ylabel("Latency (ms)")
for i, v in enumerate(latencies):
    axes[0].text(i, v + 10, f"{v:.0f} ms", ha="center", fontsize=9)

axes[1].bar(labels, tps_vals, color=colors)
axes[1].set_title("Throughput (tokens/second)")
axes[1].set_ylabel("tok/s")
for i, v in enumerate(tps_vals):
    axes[1].text(i, v + 0.5, f"{v:.1f} tok/s", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

for r in latency_results:
    rprint(f"  {r.prompt_label}: [bold]{r.total_ms:.0f} ms[/bold], {r.tokens_generated} tokens, {r.tokens_per_second:.1f} tok/s")

In [ ]:
import random
random.seed(42)

agent_steps = [
    {"label": "Plan task",         "llm_ms": 1200, "tool_ms": 0},
    {"label": "Fetch data (tool)",  "llm_ms":  900, "tool_ms": 450},
    {"label": "Analyse results",    "llm_ms": 1400, "tool_ms": 0},
    {"label": "Calculate (tool)",   "llm_ms":  800, "tool_ms": 200},
    {"label": "Generate answer",    "llm_ms": 1100, "tool_ms": 0},
]

steps      = [s["label"] for s in agent_steps]
llm_times  = [s["llm_ms"] for s in agent_steps]
tool_times = [s["tool_ms"] for s in agent_steps]
total_ms   = sum(llm_times) + sum(tool_times)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(steps))
ax.bar(x, llm_times,  label="LLM call",      color=sns.color_palette("muted")[0])
ax.bar(x, tool_times, bottom=llm_times,       label="Tool execution", color=sns.color_palette("muted")[1])
ax.set_xticks(list(x))
ax.set_xticklabels(steps, rotation=10, ha="right")
ax.set_ylabel("Time (ms)")
ax.set_title(f"Per-step latency breakdown, total: {total_ms:,} ms ({total_ms/1000:.1f} s)", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

rprint(f"Total agent task latency: [bold]{total_ms:,} ms ({total_ms/1000:.1f} s)[/bold]")
rprint(f"  LLM calls: {sum(llm_times):,} ms  |  Tool execution: {sum(tool_times):,} ms")

<a id='section-6'></a>
## 6. Compute Footprint: Memory & Context Management

### Two distinct memory concerns

1. **Process RAM**: the RAM consumed by the Python process running the agent. This covers: the LangGraph state graph, the full message history list, tool outputs, and any in-memory vector stores or caches.

2. **Context window as a memory budget**: every LLM call is bounded by a maximum context length (e.g., 128K tokens for Granite 4). Sending a large context window increases both latency and cost. Think of it as a shared resource that must be actively managed.

For API-based deployments (watsonx.ai), concern #1 is usually modest (a few hundred MB). Concern #2 is the primary operational lever.

> For self-hosted models, a third concern emerges: **GPU VRAM**. A 7B-parameter model at FP16 requires ~14 GB VRAM just for weights, before accounting for the KV cache (which grows with context length). This notebook focuses on the API-based scenario, but the context management strategies below apply equally to self-hosted deployments.

<a id='section-7'></a>
## 7. Environmental Footprint

Energy consumption and carbon emissions are increasingly important considerations for AI systems. While this is a rapidly evolving area, some grounded estimates are available:

### Energy per query

- A single LLM text query requires approximately **0.001–0.003 kWh** of energy (Google Cloud, 2025).
- For comparison: a Google web search uses ~0.0003 kWh; running a microwave for one minute uses ~0.02 kWh.
- An AI agent completing a 5-step task therefore uses **0.005–0.015 kWh** - roughly the same as sending 10–50 web search queries.

We only touch upon this topic as a means to introduce you to the consideration: the methodology for estimating per-query emissions and the variables that dominate the result. To go deeper, the resources below are a good starting point.

### Further reading

- [MIT Technology Review: AI Energy Footprint (May 2025)](https://www.technologyreview.com/2025/05/20/1116327/ai-energy-usage-climate-footprint-big-tech/)
- [Google Cloud: Measuring the environmental impact of AI inference (2025)](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference/)
- For **production monitoring** of all NFRs (cost, latency, and usage) in a single dashboard, see the companion recipe: [Observability & Tracing with Langfuse](../Tracing/Tracing_Agent.ipynb).

<a id='section-8'></a>
## 8. Summary & Reusable NFR Profiler

### Key takeaways

| Dimension | Primary metric | Biggest risk for agents | Top mitigation |
|---|---|---|---|
| **Cost** | USD per task | Context accumulation across turns | Prompt compression, model routing |
| **Latency** | ms per turn | Sequential tool calls stack up | Smaller models, parallel tool calls |
| **Throughput** | tokens/second | Blocking on long-running steps | Streaming, async execution |
| **Process memory** | MB RAM | Long message histories | Sliding window history |
| **Context budget** | % of window | Unbounded conversation history | Token-aware trimming |
| **Environmental** | kWh / CO₂e | Scale × unnecessary calls | Caching, right-sizing, routing |

### The NFR profiler

The cell below assembles all measurements from this notebook into a single reusable `NFRProfiler` class. Attach it to any LangChain LLM to collect cost, latency, and memory overhead on every run, then visualize results as a compact dashboard.

In [ ]:
@dataclass
class NFRProfiler:
    """
    Attach to any LangChain LLM to collect cost, latency, and memory per run.

    Usage:
        profiler = NFRProfiler(llm=llm, cost_estimator=SMALL_MODEL)
        profiler.run("Your prompt here")
        profiler.report()
    """
    llm: Any
    cost_estimator: TokenCostEstimator
    _results: list = field(default_factory=list)

    def run(self, prompt: str) -> dict:
        """Run the LLM and record NFR metrics."""
        mem_before = get_memory_mb()
        cb = UsageMetadataCallbackHandler()
        start = time.perf_counter()

        self.llm.invoke(prompt, config={"callbacks": [cb]})
        input_tokens  = sum(v.get("input_tokens", 0)  for v in cb.usage_metadata.values())
        output_tokens = sum(v.get("output_tokens", 0) for v in cb.usage_metadata.values())

        elapsed_ms = (time.perf_counter() - start) * 1000
        mem_after  = get_memory_mb()

        cost_info = self.cost_estimator.estimate(input_tokens, output_tokens)

        record = {
            "prompt_preview":    prompt[:50] + ("..." if len(prompt) > 50 else ""),
            "input_tokens":      input_tokens,
            "output_tokens":     output_tokens,
            "total_tokens":      input_tokens + output_tokens,
            "cost_usd":          cost_info["estimated_cost_usd"],
            "latency_ms":        round(elapsed_ms, 1),
            "memory_delta_mb":   round(mem_after - mem_before, 2),
            "tokens_per_second": round(output_tokens / max(elapsed_ms / 1000, 0.001), 1),
        }
        self._results.append(record)
        return record

    def summary_df(self) -> pd.DataFrame:
        return pd.DataFrame(self._results)

    def report(self) -> None:
        """Print a summary table and render a multi-metric dashboard."""
        df = self.summary_df()
        if df.empty:
            rprint("No runs recorded yet.")
            return

        rprint("\n[bold]NFR Profiler: Run Summary[/bold]")
        rprint(df[["prompt_preview", "total_tokens", "cost_usd",
                   "latency_ms", "tokens_per_second", "memory_delta_mb"]].to_string(index=False))

        agg = df.agg({"total_tokens": "mean", "cost_usd": "sum",
                      "latency_ms": "mean", "tokens_per_second": "mean"})
        rprint(f"\n  Avg tokens/run:  {agg['total_tokens']:.0f}")
        rprint(f"  Total cost:      ${agg['cost_usd']:.6f}")
        rprint(f"  Avg latency:     {agg['latency_ms']:.0f} ms")
        rprint(f"  Avg throughput:  {agg['tokens_per_second']:.1f} tok/s")

        fig, axes = plt.subplots(2, 2, figsize=(12, 7))
        labels = [f"Run {i+1}" for i in range(len(df))]

        axes[0, 0].bar(labels, df["total_tokens"],      color=sns.color_palette("muted")[0])
        axes[0, 0].set_title("Total tokens per run");   axes[0, 0].set_ylabel("Tokens")

        axes[0, 1].bar(labels, df["cost_usd"],          color=sns.color_palette("muted")[1])
        axes[0, 1].set_title("Estimated cost (USD)");   axes[0, 1].set_ylabel("USD")

        axes[1, 0].bar(labels, df["latency_ms"],        color=sns.color_palette("muted")[2])
        axes[1, 0].set_title("Latency per run (ms)");   axes[1, 0].set_ylabel("ms")

        axes[1, 1].bar(labels, df["tokens_per_second"], color=sns.color_palette("muted")[3])
        axes[1, 1].set_title("Throughput (tok/s)");     axes[1, 1].set_ylabel("tok/s")

        plt.suptitle("NFR Profiler Dashboard", fontsize=14, y=1.01)
        plt.tight_layout()
        plt.show()

print("NFRProfiler loaded.")

In [ ]:
profiler = NFRProfiler(
    llm=llm,
    cost_estimator=SMALL_MODEL,
)

demo_prompts = [
    "What is 9 multiplied by 7?",
    "Summarise the main tradeoffs between synchronous and asynchronous agent execution.",
    "Explain the concept of token economics for AI agents and describe three strategies to reduce cost in a production agent pipeline.",
    "You are a senior AI engineer. A business stakeholder asks why their AI agent is costing 10× more than expected after going to production. Walk through the most likely causes: context accumulation, model tier selection, tool call overhead. Provide concrete recommendations for each.",
]

for prompt in demo_prompts:
    record = profiler.run(prompt)
    rprint(f"  [green]✓[/green] {record['prompt_preview']}: {record['total_tokens']} tokens, ${record['cost_usd']:.6f}, {record['latency_ms']:.0f} ms")

profiler.report()

## Conclusion

Non-functional requirements are not optional extras for AI agents. They determine whether a system is viable in production.

The key insight from this notebook is that **agents amplify NFR risks** relative to single-turn LLM calls:

- Token costs grow super-linearly due to context re-injection on each turn.
- Latency compounds across sequential steps and tool calls.
- Memory and context budgets require active management to avoid hitting limits.

The good news is that all of these are **measurable and manageable** with the tools shown here:

| Tool / Pattern | What it addresses |
|---|---|
| `TokenCostEstimator` | Model cost awareness before deployment |
| `get_openai_callback` | Real-time token and cost instrumentation |
| `compress_history` | Input token reduction via history pruning |
| `route_to_model` | Cost reduction via dynamic model selection |
| `measure_latency` | Per-step performance benchmarking |
| `NFRProfiler` | Unified cost + latency + memory dashboard |

For production deployments, complement these patterns with an observability layer - see the [Observability & Tracing notebook](../Tracing/Tracing_Agent.ipynb) for end-to-end monitoring with Langfuse.

### References

- [IBM Granite Models](https://www.ibm.com/granite)
- [Token Economics for LLM Agents (arXiv, May 2025)](https://arxiv.org/html/2605.09104v1)
- [FinOps for AI Overview](https://www.finops.org/wg/finops-for-ai-overview/)
- [AI Cost Observability, Truefoundry (2025)](https://www.truefoundry.com/blog/ai-cost-observability)
- [Google Cloud: Measuring AI inference environmental impact (2025)](https://cloud.google.com/blog/products/infrastructure/measuring-the-environmental-impact-of-ai-inference/)
- [MIT Technology Review: AI Energy Footprint (May 2025)](https://www.technologyreview.com/2025/05/20/1116327/ai-energy-usage-climate-footprint-big-tech/)
- [LangChain Token Counting Callback (2025)](https://changelog.langchain.com/announcements/universal-token-counting-callback-for-langchain-python)